# 실습 1 · 미니 LLMB: 논문에서 셀 레코드 만들기

LLM으로 논문 속 숫자를 **믿을 수 있는 레코드**로 바꾼다. 강의 1부의 LLMB를 축소한 실습이다.

**하는 일**
1. 미니 논문 3편에서 물성값을 뽑는다 (스키마 v1) → 채점하면 틀린다
2. 스키마를 v2로 바꿔 **근거·측정 여부·포메이션 여부**를 같이 뽑고, 코드로 걸러 고친다
3. 프롬프트를 직접 써서(도메인 지식 + 출력 형식) **LLM 출력부터** 맞게 만든다
4. 그림 범례와 본문 셀 이름을 맞춰 **셀 단위 표**를 만든다

**바꾸는 값**: 3회차 `PROMPT`, 7번의 `MATCH`와 `MERGE_PROMPT`. 나머지는 실행만 한다.

> 미니 논문은 실습용으로 작성한 **가상 논문**이다. 함정을 일부러 넣어 두었다: 다른 논문에서 인용한 값, 포메이션 사이클 값, S/m 단위, 범례 ≠ 셀 이름.
> API 호출: 한 번 끝까지 실행하면 약 15회, 3회차 프롬프트를 고쳐 다시 실행할 때마다 3회 더. 같은 입력은 저장해 두었다가 다시 쓰므로, 같은 프롬프트로 다시 실행하면 호출이 늘지 않는다.

In [ ]:
#@title 1. 환경 설정 — 한 번만 실행 (펼치면 도우미 코드)

# 이 셀은 실습에 필요한 도우미 함수를 정의한다. 고칠 필요는 없다.

# ── 공통 도우미: 환경 확인 · API 키 · Gemini 호출 · 캐시 · 속도 제한 ──────────────
# Colab, 로컬 Jupyter, VS Code 어디서나 같은 코드로 돈다.
import hashlib
import importlib
import json
import os
import pathlib
import subprocess
import sys
import time

IN_COLAB = "google.colab" in sys.modules
MODEL = "gemini-3.5-flash-lite"            # 막히면 FALLBACK_MODEL로 자동 전환
FALLBACK_MODEL = "gemini-3.1-flash-lite"
EMBED_MODEL = "gemini-embedding-001"       # 막히면 "tfidf"로 바꿔 API 없이 진행
RPM = 10                                   # 무료 티어 분당 호출 한도에 맞춘 간격
WORK = pathlib.Path("/content/llm_practice" if IN_COLAB else "llm_practice_work")
_client = None
_last_call = [0.0]


def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


def ensure_packages(extra=()):
    """필요한 패키지가 없으면 설치한다 (Colab·로컬 공통)."""
    import importlib.metadata as md
    import importlib.util
    need = {"google.genai": "google-genai>=2.20,<3", "pydantic": "pydantic>=2", "pandas": "pandas",
            "matplotlib": "matplotlib", "numpy": "numpy"}
    need.update(dict(extra))
    missing = []
    for mod, pkg in need.items():
        try:
            found = importlib.util.find_spec(mod) is not None
        except ModuleNotFoundError:
            found = False
        if not found:
            missing.append(pkg)
    try:                                   # Colab에 깔린 google-genai가 오래됐으면 올린다
        if int(md.version("google-genai").split(".")[0]) < 2 and "google-genai>=2.20,<3" not in missing:
            missing.append("google-genai>=2.20,<3")
    except md.PackageNotFoundError:
        pass
    if missing:
        print("설치 중:", ", ".join(missing))
        _pip(missing)
        importlib.invalidate_caches()
    # 기관·회사 네트워크의 인증서 검사 대응 (있으면 쓰고, 없으면 넘어간다)
    try:
        import truststore
        truststore.inject_into_ssl()
    except Exception:
        pass


def _read_dotenv():
    for d in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        f = d / ".env"
        if f.is_file():
            for line in f.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if line.startswith(("GEMINI_API_KEY=", "GOOGLE_API_KEY=")):
                    return line.split("=", 1)[1].strip().strip("'\"")
    return None


def load_api_key():
    """환경변수 → .env 파일 → Colab 보안 비밀 → 직접 입력 순서로 찾는다."""
    key = os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    src = "환경변수"
    if not key:
        key, src = _read_dotenv(), ".env 파일"
    if not key and IN_COLAB:
        try:
            from google.colab import userdata
            key, src = userdata.get("GEMINI_API_KEY"), "Colab 보안 비밀"
        except Exception:
            key = None
    if not key:
        import getpass
        key, src = getpass.getpass("Gemini API 키를 붙여 넣으세요 (화면에 보이지 않음): ").strip(), "직접 입력"
    if not key:
        raise RuntimeError("API 키를 찾지 못했다. README의 'API 키 준비'를 따라 키를 등록한다.")
    os.environ["GEMINI_API_KEY"] = key
    return key, src


def gemini_client():
    """키를 찾아 Gemini 클라이언트를 만든다. '키 확인: 찾음'이 나오면 된다."""
    global _client
    from google import genai
    from google.genai import types
    key, src = load_api_key()
    _client = genai.Client(api_key=key, http_options=types.HttpOptions(
        retry_options=types.HttpRetryOptions(attempts=4, initial_delay=8.0)))
    print(f"키 확인: 찾음 ({src}, …{key[-4:]}) · 모델 {MODEL}")
    return _client


def _cache_path():
    WORK.mkdir(parents=True, exist_ok=True)
    return WORK / "llm_cache.jsonl"


def _cache_load():
    p, out = _cache_path(), {}
    if p.exists():
        for line in p.read_text(encoding="utf-8").splitlines():
            try:
                d = json.loads(line)
                out[d["k"]] = d["v"]
            except Exception:
                pass
    return out


_CACHE = None


def _cached(key):
    global _CACHE
    if _CACHE is None:
        _CACHE = _cache_load()
    return _CACHE.get(key)


def _store(key, value):
    _CACHE[key] = value
    with open(_cache_path(), "a", encoding="utf-8") as f:
        f.write(json.dumps({"k": key, "v": value}, ensure_ascii=False) + "\n")


def _throttle():
    gap = 60.0 / max(RPM, 1)
    wait = _last_call[0] + gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()


def llm(prompt, schema=None, image=None, model=None):
    """Gemini 호출 한 번. schema(pydantic 모델)를 주면 그 형식의 객체를, 아니면 글자를 돌려준다.
    같은 입력은 캐시에서 꺼내므로 다시 실행해도 호출 수가 늘지 않는다."""
    from google.genai import types
    if _client is None:
        gemini_client()
    model = model or MODEL
    h = hashlib.sha256()
    h.update((model + "\n" + prompt).encode("utf-8"))
    if schema is not None:
        h.update(json.dumps(schema.model_json_schema(), sort_keys=True).encode())
    if image is not None:
        h.update(image)
    key = h.hexdigest()
    hit = _cached(key)
    if hit is None:
        contents = [types.Part.from_bytes(data=image, mime_type="image/png"), prompt] if image is not None else prompt
        cfg = None
        if schema is not None:
            cfg = types.GenerateContentConfig(response_mime_type="application/json",
                                              response_json_schema=schema.model_json_schema())
        last_err = None
        for m in (model, FALLBACK_MODEL):
            try:
                _throttle()
                resp = _client.models.generate_content(model=m, contents=contents, config=cfg)
                hit = resp.text
                break
            except Exception as e:                       # 쿼터 소진 등 → 대체 모델
                last_err = e
                print(f"  ! {m} 호출 실패 ({type(e).__name__}): {str(e)[:120]}")
        if hit is None:
            raise RuntimeError(f"모델 호출 실패: {last_err}")
        _store(key, hit)
    return schema.model_validate_json(hit) if schema is not None else hit


def embed_texts(texts, model=None):
    """문장 목록을 벡터로 바꾼다. model='tfidf'면 API 없이 TF-IDF로 대신한다."""
    import numpy as np
    model = model or EMBED_MODEL
    if model == "tfidf":
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(stop_words="english").fit(texts)
        return np.asarray(vec.transform(texts).todense()), vec
    if _client is None:
        gemini_client()
    out = []
    for i in range(0, len(texts), 50):                 # 한 번에 최대 50개
        batch = texts[i:i + 50]
        key = hashlib.sha256((model + "\n" + "\x1e".join(batch)).encode("utf-8")).hexdigest()
        hit = _cached(key)
        if hit is None:
            _throttle()
            r = _client.models.embed_content(model=model, contents=batch)
            hit = [e.values for e in r.embeddings]
            _store(key, hit)
        out.extend(hit)
    return np.asarray(out, dtype="float32"), None


def save_json(name, obj):
    WORK.mkdir(parents=True, exist_ok=True)
    p = WORK / name
    p.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")
    return p


def download(paths, zip_name):
    """결과 파일을 zip으로 묶는다. Colab이면 내려받기 창을 띄운다."""
    import zipfile
    WORK.mkdir(parents=True, exist_ok=True)
    zp = WORK / zip_name
    with zipfile.ZipFile(zp, "w") as z:
        for p in paths:
            p = pathlib.Path(p)
            if p.exists():
                z.write(p, p.name)
    print("저장:", zp.resolve())
    if IN_COLAB:
        from google.colab import files
        files.download(str(zp))
    return zp


ensure_packages({})

import warnings; warnings.filterwarnings('ignore')

import matplotlib; matplotlib.rcParams['figure.dpi'] = 100

# ── 미니 논문 3편 (실습용으로 작성한 가상 논문, 실제 논문 아님) ─────────────────────
# 실습 1의 추출·채점 대상이고, 실습 2에서는 Europe PMC에 접속할 수 없을 때 대신 쓰는 말뭉치다.
PAPERS = {
    "A": {
        "title": "Weakly solvating ether electrolytes for stable Li||NCM811 cells",
        "methods": (
            "Electrolytes were prepared by dissolving 1.0 M LiFSI in diethyl ether (DEE), dipropyl ether (DPE) or "
            "diethylene glycol dimethyl ether (DEGDME); the corresponding cells are denoted as the DEE-based cell, the "
            "DPE-based cell and the baseline cell. CR2032 coin cells were assembled with 50 µm Li foil, NCM811 cathodes "
            "(areal capacity 2.0 mAh cm−2), a Celgard 2325 separator and 40 µL of electrolyte. After two formation cycles "
            "at 0.1C, the cells were cycled at 1C between 2.8 and 4.4 V at 25 °C. Ionic conductivity was measured by "
            "electrochemical impedance spectroscopy at 25 °C."),
        "results": (
            "The DEE electrolyte showed an ionic conductivity of 4.2 mS cm−1, lower than that of the DEGDME electrolyte "
            "(7.9 mS cm−1). A previous study reported 10.6 mS cm−1 for a localized high-concentration electrolyte [21]. "
            "In the first formation cycle at 0.1C, the DEE-based cell delivered 205 mAh g−1. At 1C, the DEE-based cell "
            "showed an initial discharge capacity of 182 mAh g−1 and retained 94% of its capacity after 100 cycles, "
            "whereas the baseline cell retained only 66% (Fig. 2a). The average Coulombic efficiency of the DEE-based "
            "cell over 100 cycles was 99.6%."),
        "caption": (
            "Fig. 2 | Cycling performance of Li||NCM811 cells at 1C after formation. a, Discharge capacity (filled "
            "symbols, left axis) and Coulombic efficiency (open symbols, right axis). E1, E2 and Ref. denote the DEE, "
            "DPE and DEGDME electrolytes, respectively."),
    },
    "B": {
        "title": "A nitrate additive for dendrite-free Li||LFP cells",
        "methods": (
            "The baseline electrolyte was 1.0 M LiTFSI in a 1:1 (v/v) mixture of 1,3-dioxolane (DOL) and "
            "1,2-dimethoxyethane (DME). For the LiNO3-added electrolyte, 2 wt% LiNO3 was added to the baseline "
            "electrolyte. Li||LiFePO4 coin cells were cycled at 0.5C between 2.5 and 4.0 V."),
        "results": (
            "The LiNO3-added electrolyte exhibited an ionic conductivity of 1.05 S m−1, comparable to that of the "
            "additive-free electrolyte (1.12 S m−1). The LiNO3-added cell delivered an initial discharge capacity of "
            "152 mAh g−1 at 0.5C and retained 97% of its capacity after 200 cycles, while the additive-free cell "
            "retained 71%. Previous reports on LiNO3-containing carbonate electrolytes showed a retention of about "
            "80% [12]."),
        "caption": "Fig. 3 | Cycling stability of Li||LFP cells with and without the LiNO3 additive at 0.5C.",
    },
    "C": {
        "title": "Localized high-concentration electrolyte for 4.5 V Li metal cells",
        "methods": (
            "The localized high-concentration electrolyte (LHCE) consisted of LiFSI, DME and TTE in a 1:1.2:3 molar "
            "ratio, and the conventional electrolyte was 1.0 M LiPF6 in EC/DMC (1:1, v/v). Li||NCM622 cells were "
            "cycled between 3.0 and 4.5 V at 1C after one formation cycle at 0.1C."),
        "results": (
            "The ionic conductivity of the LHCE (1.6 mS cm−1) was lower than that of the conventional electrolyte "
            "(10.1 mS cm−1). The first-cycle discharge capacity of the LHCE cell at 0.1C was 198 mAh g−1, and its "
            "initial discharge capacity at 1C was 176 mAh g−1. After 300 cycles, the LHCE cell retained 89% of its "
            "initial capacity, whereas the conventional cell failed after 120 cycles. The LHCE cell maintained an "
            "average Coulombic efficiency of 99.7%, compared with 98.1% for the conventional cell."),
        "caption": "Fig. 4 | Long-term cycling of Li||NCM622 cells with the LHCE and the conventional electrolyte.",
    },
}


def paper_text(pid):
    p = PAPERS[pid]
    return f"Title: {p['title']}\n\nMethods: {p['methods']}\n\nResults: {p['results']}\n\nCaption: {p['caption']}"


def show_papers():
    for pid in PAPERS:
        print(f"[논문 {pid}] " + paper_text(pid).replace("\n\n", "\n") + "\n")


# ── 실습 1 · 미니 LLMB: 논문에서 셀 레코드 만들기 ───────────────────────────────
# 미니 논문 3편은 mini_papers.py에 있다 (실습용 가상 논문).
# 일부러 넣은 함정: 다른 논문에서 인용한 값, 포메이션 사이클 값, S/m 단위, 범례 ≠ 셀 이름.
import json
import re
from typing import Literal, Optional

from pydantic import BaseModel, Field

# 셀(또는 전해질) 이름 → 셀 ID. 채점할 때 '같은 셀'인지 판단하는 데 쓴다.
ALIASES = {
    "A-DEE": ["dee-based cell", "dee electrolyte", "diethyl ether", "dee", "e1"],
    "A-DPE": ["dpe-based cell", "dpe electrolyte", "dipropyl ether", "dpe", "e2"],
    "A-REF": ["baseline cell", "degdme electrolyte", "degdme", "baseline", "ref"],
    "B-NO3": ["lino3-added cell", "lino3-added electrolyte", "lino3-added", "lino3"],
    "B-FREE": ["additive-free cell", "additive-free electrolyte", "additive-free", "baseline electrolyte", "baseline"],
    "C-LHCE": ["lhce cell", "lhce", "localized high-concentration electrolyte (lhce)"],
    "C-CONV": ["conventional cell", "conventional electrolyte", "conventional", "lipf6", "ec/dmc"],
}

# 정답: 이 논문에서 측정한 값만, 표준 단위로 (mS/cm, mAh/g, %)
GOLD = [
    ("A", "A-DEE", "ionic_conductivity", 4.2), ("A", "A-REF", "ionic_conductivity", 7.9),
    ("A", "A-DEE", "initial_discharge_capacity", 182), ("A", "A-DEE", "capacity_retention", 94),
    ("A", "A-REF", "capacity_retention", 66), ("A", "A-DEE", "average_coulombic_efficiency", 99.6),
    ("B", "B-NO3", "ionic_conductivity", 10.5), ("B", "B-FREE", "ionic_conductivity", 11.2),
    ("B", "B-NO3", "initial_discharge_capacity", 152), ("B", "B-NO3", "capacity_retention", 97),
    ("B", "B-FREE", "capacity_retention", 71),
    ("C", "C-LHCE", "ionic_conductivity", 1.6), ("C", "C-CONV", "ionic_conductivity", 10.1),
    ("C", "C-LHCE", "initial_discharge_capacity", 176), ("C", "C-LHCE", "capacity_retention", 89),
    ("C", "C-LHCE", "average_coulombic_efficiency", 99.7), ("C", "C-CONV", "average_coulombic_efficiency", 98.1),
]
CANON_UNIT = {"ionic_conductivity": "mS/cm", "initial_discharge_capacity": "mAh/g",
              "capacity_retention": "%", "average_coulombic_efficiency": "%"}

# ── 스키마: v1(값만) → v2(근거·측정 여부·조건 먼저) ─────────────────────────────
PROPS = Literal["ionic_conductivity", "initial_discharge_capacity", "capacity_retention",
                "average_coulombic_efficiency", "other"]


class RecV1(BaseModel):
    material: str = Field(description="셀 또는 전해질 이름")
    property: PROPS
    value: Optional[float]
    unit: Optional[str]


class OutV1(BaseModel):
    records: list[RecV1]


class RecV2(BaseModel):
    evidence: str = Field(description="값이 나온 원문 문장을 그대로 복사")
    this_work: bool = Field(description="이 논문에서 측정한 값이면 true, 다른 논문에서 인용한 값이면 false")
    formation_cycle: bool = Field(description="포메이션(활성화) 사이클에서 얻은 값이면 true, 본 시험 사이클 값이면 false")
    method: Optional[str] = Field(description="측정 방법 (예: EIS, galvanostatic cycling)")
    condition: Optional[str] = Field(description="측정 조건 (C-rate, 사이클 수, 온도 등)")
    material: str = Field(description="셀 또는 전해질 이름")
    property: PROPS
    value: Optional[float]
    unit: Optional[str]


class OutV2(BaseModel):
    records: list[RecV2]


SCHEMAS = {"v1": OutV1, "v2": OutV2}
PROP_HELP = ("ionic_conductivity = 전해질 이온전도도, initial_discharge_capacity = 초기 방전 용량, "
             "capacity_retention = 용량 유지율, average_coulombic_efficiency = 평균 쿨롱효율, 그 밖의 값 = other")


# ── 추출 ────────────────────────────────────────────────────────────────
def extract(papers, schema="v1", prompt=""):
    """논문마다 LLM을 한 번 불러 레코드를 뽑는다."""
    model = SCHEMAS[schema]
    out = []
    for pid in papers:
        ask = ("Extract every numerical property value reported in the paper below as JSON records.\n"
               f"Property names: {PROP_HELP}.\n"
               + (f"Additional rules:\n{prompt.strip()}\n" if prompt.strip() else "")
               + "\nPAPER:\n" + paper_text(pid))
        res = llm(ask, schema=model)
        for r in res.records:
            d = r.model_dump()
            d["paper"] = pid
            out.append(d)
    print(f"추출: 논문 {len(papers)}편 → 레코드 {len(out)}개 (스키마 {schema})")
    return out


# ── 채점 ────────────────────────────────────────────────────────────────
def _norm(s):
    return re.sub(r"[^a-z0-9/+.()\- ]", "", (s or "").lower()).strip()


def resolve_cell(paper, name):
    """셀 이름을 정답 셀 ID로 바꾼다 (같은 논문 안에서 가장 길게 일치하는 별칭)."""
    n = _norm(name)
    best, blen = None, 0
    for cid, als in ALIASES.items():
        if not cid.startswith(paper + "-"):
            continue
        for a in als:
            if a in n and len(a) > blen:
                best, blen = cid, len(a)
    return best


def _close(a, b, rel=0.02):
    return a is not None and b is not None and abs(a - b) <= max(abs(b) * rel, 0.05)


def score(records, gold=None, show=True, label=None):
    """정답과 비교해 정밀도·재현율·F1을 낸다. 표(merge 결과)를 주면 표 채점을 한다.
    label을 주면 그 이름으로 기록해 show_scores()에서 회차끼리 비교한다."""
    import pandas as pd
    gold = GOLD if gold is None else gold
    if gold is GOLD_TABLE:
        return _score_table(records, show)
    used, rows = set(), []
    other = [r for r in records if r.get("property") not in CANON_UNIT]
    records = [r for r in records if r.get("property") in CANON_UNIT]   # 'other'는 채점 대상이 아님
    for r in records:
        cid = resolve_cell(r["paper"], r.get("material"))
        val = r.get("value_std", r.get("value"))
        hit = None
        for gi, (gp, gc, gprop, gv) in enumerate(gold):
            if gi in used or gp != r["paper"] or gc != cid or gprop != r.get("property"):
                continue
            if _close(val, gv):
                hit = gi
                break
        if hit is not None:
            used.add(hit)
        rows.append({"논문": r["paper"], "물질/셀": r.get("material"), "물성": r.get("property"),
                     "값": val, "단위": r.get("unit_std", r.get("unit")), "판정": "정답" if hit is not None else "오답"})
    missed = [{"논문": gp, "물질/셀": gc, "물성": gprop, "값": gv, "단위": CANON_UNIT[gprop], "판정": "놓침"}
              for gi, (gp, gc, gprop, gv) in enumerate(gold) if gi not in used]
    tp, fp, fn = len(used), len(rows) - len(used), len(missed)
    p = tp / (tp + fp) if tp + fp else 0.0
    rcl = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * rcl / (p + rcl) if p + rcl else 0.0
    summary = {"정밀도": round(p, 3), "재현율": round(rcl, 3), "F1": round(f1, 3), "정답": tp, "오답": fp, "놓침": fn}
    SCORES.append({"채점": label or f"레코드 {len(records)}개", **summary})
    if show:
        print(f"정밀도 {p:.2f} · 재현율 {rcl:.2f} · F1 {f1:.2f}  (정답 {tp}, 오답 {fp}, 놓침 {fn})"
              + (f"  · 'other' {len(other)}개는 채점 제외" if other else ""))
        bad = pd.DataFrame([x for x in rows if x["판정"] == "오답"] + missed)
        if len(bad):
            print("\n틀린 것과 놓친 것:")
            _display(bad)
    return summary


SCORES = []


def show_scores():
    """지금까지 채점한 결과를 한 표로 본다 (같은 이름은 마지막 결과만)."""
    import pandas as pd
    rows = {}
    for s in SCORES:
        if "F1" in s:
            rows[s["채점"]] = s
    _display(pd.DataFrame(list(rows.values())))


def _display(df):
    try:
        from IPython.display import display
        display(df)
    except Exception:
        print(df.to_string())


# ── 코드로 검증하기 ───────────────────────────────────────────────────────
_UNIT = {
    "ms/cm": 1.0, "mscm-1": 1.0, "s/m": 10.0, "sm-1": 10.0, "s/cm": 1000.0, "scm-1": 1000.0,
    "mah/g": 1.0, "mahg-1": 1.0, "%": 1.0,
}
_RANGE = {"ionic_conductivity": (0.01, 100), "initial_discharge_capacity": (10, 400),
          "capacity_retention": (0, 100), "average_coulombic_efficiency": (50, 100)}


def _unit_key(u):
    u = (u or "").lower().replace("−", "-").replace("⁻¹", "-1").replace(" ", "").replace("·", "")
    return u.replace("^-1", "-1")


def _evidence_ok(pid, ev):
    if not ev:
        return False
    src = re.sub(r"\s+", " ", paper_text(pid))
    ev = re.sub(r"\s+", " ", ev).strip()
    if ev[:60] in src:
        return True
    toks = set(re.findall(r"[a-z0-9.]+", ev.lower()))
    stoks = set(re.findall(r"[a-z0-9.]+", src.lower()))
    return len(toks) > 0 and len(toks & stoks) / len(toks) >= 0.9


def validate(records):
    """LLM이 아니라 코드가 확인한다: 인용값·포메이션 값 제거(v2 필드), 근거 문장 확인, 단위 환산, 값 범위."""
    out, log = [], []
    for r in records:
        r = dict(r)
        tag = f"{r['paper']}/{r.get('material')}/{r.get('property')}={r.get('value')} {r.get('unit')}"
        if r.get("this_work") is False:
            log.append(f"제거(다른 논문 값): {tag}")
            continue
        if r.get("formation_cycle") is True:
            log.append(f"제거(포메이션 사이클 값): {tag}")
            continue
        if "evidence" in r and not _evidence_ok(r["paper"], r.get("evidence")):
            log.append(f"제거(근거 문장이 원문에 없음): {tag}")
            continue
        prop = r.get("property")
        if prop in CANON_UNIT and r.get("value") is not None:
            f = _UNIT.get(_unit_key(r.get("unit")))
            if f is None:
                log.append(f"제거(모르는 단위 '{r.get('unit')}'): {tag}")
                continue
            r["value_std"], r["unit_std"] = round(r["value"] * f, 4), CANON_UNIT[prop]
            if f != 1.0:
                log.append(f"단위 환산: {tag} → {r['value_std']} {r['unit_std']}")
            lo, hi = _RANGE[prop]
            if not lo <= r["value_std"] <= hi:
                log.append(f"제거(범위 밖): {tag}")
                continue
        out.append(r)
    print(f"검증: {len(records)}개 → {len(out)}개")
    for line in log:
        print("  -", line)
    return out


# ── 범례와 셀 이름 맞추기 → 셀 단위 표 ───────────────────────────────────────
# 논문 A Fig. 2의 범례. 그래프에서 읽은 값은 범례로만 구분되므로, 본문 레코드와 합치려면 범례 → 셀 이름 연결이 필요하다.
FIG2_LEGENDS = ["E1", "E2", "Ref."]

GOLD_TABLE = [   # 전해질 칸은 염 + 농도 + 용매가 모두 있어야 정답
    {"legend": "E1", "cell": "A-DEE", "solvent": ["dee", "diethyl ether"], "cathode": "NCM811", "c_rate": "1C"},
    {"legend": "E2", "cell": "A-DPE", "solvent": ["dpe", "dipropyl ether"], "cathode": "NCM811", "c_rate": "1C"},
    {"legend": "Ref.", "cell": "A-REF", "solvent": ["degdme", "diethylene glycol dimethyl ether"], "cathode": "NCM811",
     "c_rate": "1C"},
]
GOLD_SALT, GOLD_CONC = "lifsi", ("1.0 m", "1 m")

MERGE_PROMPT_DEFAULT = """
그림의 범례 라벨을 본문에 나온 셀과 연결한다.
- 캡션에서 각 범례가 어떤 전해질을 뜻하는지 찾고, Methods에서 그 전해질을 쓴 셀 이름을 찾는다.
- cell_name에는 Methods에 나온 셀 이름을 그대로 쓴다. 찾지 못하면 null로 둔다.
- electrolyte에는 염과 용매를 쓴다 (예: 1.0 M LiFSI in DEE). cathode에는 양극 물질, c_rate에는 본 시험 C-rate를 쓴다.

범례 라벨: {legends}
추출한 레코드에 나온 이름 (참고): {names}

논문:
{paper}
"""


class LegendMatch(BaseModel):
    legend: str
    cell_name: Optional[str] = Field(description="본문에 나온 셀 이름")
    electrolyte: Optional[str]
    cathode: Optional[str]
    c_rate: Optional[str]


class LegendMatches(BaseModel):
    matches: list[LegendMatch]


_TABLE_PROPS = {"initial_discharge_capacity": "초기 용량 (mAh/g)", "capacity_retention": "유지율 (%)",
                "average_coulombic_efficiency": "평균 CE (%)", "ionic_conductivity": "전도도 (mS/cm)"}


def merge(records, match="exact", prompt=None):
    """범례(그림)와 셀 이름(본문)을 맞추고, 본문 레코드 값을 붙여 셀 단위 표를 만든다.
    match="llm"이면 prompt의 {legends}, {names}, {paper} 자리에 범례, 레코드 이름, 논문 A 원문을 넣어 LLM에 보낸다."""
    import pandas as pd
    names = sorted({r["material"] for r in records if r["paper"] == "A"})
    info = {}
    if match == "exact":
        for lg in FIG2_LEGENDS:
            hit = next((n for n in names if n.strip().lower() == lg.strip().lower()), None)
            info[lg] = {"cell_name": hit, "electrolyte": None, "cathode": None, "c_rate": None}
    else:
        ask = (prompt or MERGE_PROMPT_DEFAULT).strip()
        ask = ask.replace("{legends}", ", ".join(FIG2_LEGENDS)).replace("{names}", ", ".join(names))
        src = paper_text("A")
        print("LLM에 보낸 프롬프트 ─────────────\n"
              + ask.replace("{paper}", f"[논문 A 원문 {len(src)}자]") + "\n───────────────────────────\n")
        ask = ask.replace("{paper}", src)
        for m in llm(ask, schema=LegendMatches).matches:
            info[m.legend.strip()] = m.model_dump()
    rows = []
    for lg in FIG2_LEGENDS:
        i = info.get(lg, {})
        row = {"범례": lg, "셀 (본문)": i.get("cell_name"), "전해질": i.get("electrolyte"),
               "양극": i.get("cathode"), "C-rate": i.get("c_rate")}
        cid = resolve_cell("A", i.get("cell_name") or "")
        for prop, col in _TABLE_PROPS.items():          # 같은 셀의 본문 레코드 값을 붙인다
            vals = [r.get("value_std", r.get("value")) for r in records
                    if r["paper"] == "A" and r.get("property") == prop and cid
                    and resolve_cell("A", r.get("material")) == cid]
            row[col] = vals[0] if vals else None
        rows.append(row)
    df = pd.DataFrame(rows)
    matched = df["셀 (본문)"].notna().sum()
    print(f"매칭 방식 '{match}': 범례 {len(df)}개 중 {matched}개를 셀 이름과 연결")
    _display(df)
    return df


def _score_table(df, show=True):
    ok, total = 0, 0
    for g in GOLD_TABLE:
        row = df[df["범례"] == g["legend"]]
        checks = {"셀": False, "전해질": False, "양극": False, "C-rate": False}
        if len(row):
            r = row.iloc[0]
            e = (r["전해질"] or "").lower()
            checks["셀"] = resolve_cell("A", r["셀 (본문)"] or "") == g["cell"]
            checks["전해질"] = (any(x in e for x in g["solvent"]) and GOLD_SALT in e
                               and any(c in e for c in GOLD_CONC))
            checks["양극"] = g["cathode"].lower() in (r["양극"] or "").lower().replace(" ", "")
            checks["C-rate"] = (r["C-rate"] or "").lower().replace(" ", "") == g["c_rate"].lower()
        ok += sum(checks.values()); total += len(checks)
        if show:
            print(f"  {g['legend']:5s} " + "  ".join(f"{k} {'O' if v else 'X'}" for k, v in checks.items()))
    acc = ok / total
    SCORES.append({"채점": "셀 단위 표", "정확도": round(acc, 3)})
    print(f"셀 단위 표 정확도 {acc:.2f} ({ok}/{total} 칸)")
    return {"정확도": round(acc, 3)}


# ── 내보내기 ────────────────────────────────────────────────────────────
def export(records, table, schema, prompt, merge_prompt=None):
    import pandas as pd
    WORK.mkdir(parents=True, exist_ok=True)
    rp = WORK / "records.csv"
    pd.DataFrame(records).to_csv(rp, index=False, encoding="utf-8-sig")
    tp = WORK / "cell_table.csv"
    table.to_csv(tp, index=False, encoding="utf-8-sig")
    sp = save_json("schema.json", {"schema": schema, "json_schema": SCHEMAS[schema].model_json_schema()})
    pp = save_json("prompt.json", {"prompt": prompt, "merge_prompt": merge_prompt, "model": MODEL})
    cp = WORK / "scores.csv"
    pd.DataFrame(SCORES).to_csv(cp, index=False, encoding="utf-8-sig")
    return download([rp, tp, sp, pp, cp], "p1_results.zip")


print('준비 완료 ·', 'Colab' if IN_COLAB else '로컬', '· 작업 폴더', WORK.resolve())

## 2. Gemini API 키 확인

키는 **각자 발급**해서 쓴다. 발급: <https://aistudio.google.com/apikey> → **Create API key** (개인 구글 계정, 무료).

| 환경 | 키를 넣는 곳 |
|---|---|
| Colab | 왼쪽 **열쇠 아이콘(보안 비밀)** → 이름 `GEMINI_API_KEY`, 값에 키 → **노트북 액세스** 켜기 |
| 로컬 (Jupyter, VS Code) | 노트북과 같은 폴더나 상위 폴더의 `.env` 파일에 `GEMINI_API_KEY=키` 한 줄 |
| 둘 다 없으면 | 아래 셀을 실행하면 입력창이 뜬다 (화면에 보이지 않음) |

키는 비밀번호와 같다. 셀에 직접 적거나 공유하지 않는다. `키 확인: 찾음`이 나오면 된다.

In [ ]:
client = gemini_client()           # '키 확인: 찾음'이 나와야 한다
MODEL = "gemini-3.5-flash-lite"    # 막히면 "gemini-3.1-flash-lite"

## 미니 논문 읽어 보기

먼저 사람이 읽어 보자. 논문 A에서 **이 논문이 측정한 값**은 몇 개인가? 인용한 값, 포메이션 사이클 값은 어디 있나?

In [ ]:
show_papers()

## 3. 스키마 정하기 — 무엇을 꺼낼지

| 스키마 | 필드 |
|---|---|
| `v1` | material, property, value, unit |
| `v2` | **evidence**(근거 문장), **this_work**(이 논문에서 측정했나), **formation_cycle**(포메이션 사이클 값인가), method, condition, material, property, value, unit |

**1회차는 `v1`** 으로 실행한다. 2회차는 아래 2회차 셀에 `v2`로 바꿔 두었다.

In [ ]:
SCHEMA = "v1"   # 1회차

## 4. 추출 프롬프트 — 어떻게 꺼낼지

**1회차는 비워 둔 채** 실행한다. 비어 있어도 기본 지시(물성 이름과 뜻)는 들어간다.
2회차는 규칙 두 줄을 넣어 두었고, 3회차는 직접 쓴다.

In [ ]:
PROMPT = """
"""

## 5. 추출하고 채점하기

논문마다 LLM을 한 번 부른다. 채점은 정답 17개와 비교한 **정밀도**(뽑은 것 중 맞은 비율), **재현율**(뽑아야 할 것 중 뽑은 비율), **F1**이다.

In [ ]:
records = extract(PAPERS, SCHEMA, PROMPT)
score(records, GOLD, label="1회차 · LLM 출력")   # 정밀도, 재현율, F1

### 무엇이 틀렸나?

틀린 것 표를 보고 원인을 적어 보자.
- 다른 논문에서 **인용한 값**이 들어왔나? (논문 A의 10.6, 논문 B의 80%)
- **포메이션 사이클**(0.1C) 값이 초기 용량으로 들어왔나? (205, 198)
- 논문 B의 이온전도도는 왜 놓쳤나? (**S m⁻¹** 단위 — 표준 단위는 mS/cm)

## 6. 코드로 검증하기 — `records` → `checked`

LLM이 뽑은 표(`records`)를 **파이썬 코드가 한 줄씩 검사**해, 틀린 줄은 지우고 단위는 통일한다. 남은 표가 `checked`다.
- 사람이 결과표를 훑으며 "이건 아니네" 하고 지우는 일을 규칙으로 만든 것
- LLM을 다시 부르지 않는다 → 호출 0회, 몇 번 돌려도 결과가 같다

**검사 다섯 가지** (위에서부터 차례로)

| 검사 | 무엇을 보나 | 처리 | v1 | v2 |
|---|---|---|---|---|
| 인용값 | `this_work`가 false인가 | 삭제 | 못 함 (필드 없음) | 함 |
| 포메이션 값 | `formation_cycle`이 true인가 | 삭제 | 못 함 (필드 없음) | 함 |
| 근거 문장 | `evidence`가 원문에 실제로 있는가 | 없으면 삭제 (지어낸 값) | 못 함 (필드 없음) | 함 |
| 단위 | S m⁻¹, S cm⁻¹ → mS/cm | 환산 | 함 | 함 |
| 값 범위 | 예) 용량 10–400 mAh/g, 효율 50–100% | 벗어나면 삭제 | 함 | 함 |

**예시 — 레코드가 검증을 지나면**

| LLM이 뽑은 레코드 | v1 검증 | v2 검증 |
|---|---|---|
| DEE-based cell · 초기 용량 205 mAh g⁻¹ (0.1C 포메이션) | 통과 → **오답으로 남음** | 삭제 (`formation_cycle`) |
| localized high-concentration electrolyte · 10.6 mS cm⁻¹ (인용) | 통과 → **오답으로 남음** | 삭제 (`this_work`) |
| LiNO3-added electrolyte · 1.05 S m⁻¹ | 10.5 mS/cm로 환산 → 정답 | 10.5 mS/cm로 환산 → 정답 |
| DEE electrolyte · 4.2 mS cm⁻¹ | 통과 | 통과 |

**알아 둘 것**
- 코드는 **레코드에 적힌 정보로만** 판단한다 → v1은 인용·포메이션 여부를 적지 않아서 거를 수 없다. 205, 10.6은 값 범위도 정상이라 그대로 통과한다
- 코드는 **지우고 환산만** 한다 → 틀린 값을 맞는 값으로 채우지는 못한다
- LLM이 필드를 잘못 적으면(인용값인데 `this_work=true`) 코드는 그대로 믿는다

아래 셀을 실행하고 로그의 `제거(...)`, `단위 환산` 줄과 **검증 뒤에도 남은 오답**을 확인한다. 왜 남았나?

In [ ]:
checked = validate(records)
score(checked, GOLD, label="1회차 · 코드 검증 후")

### 2회차 — 스키마 v2: 필드를 더 받아 코드로 거른다

아래 셀에 `SCHEMA = "v2"`와 규칙 두 줄을 넣어 두었다. 차례로 실행하고 **LLM 출력**과 **코드 검증 후** 점수를 비교한다.
- v2는 값과 함께 근거 문장, `this_work`, `formation_cycle`을 뽑는다
- LLM 출력 자체는 1회차와 비슷하게 틀린다 → 코드가 이 필드를 보고 걸러낸다
- 검증 로그에서 무엇이 왜 제거·환산됐는지 확인한다

In [ ]:
SCHEMA = "v2"   # 2회차
PROMPT = """
원문에 명시된 값만 추출한다. 값, 단위, 조건을 원문 그대로 보존한다.
다른 논문에서 인용한 값은 제외한다. 없으면 null로 둔다.
"""

In [ ]:
records = extract(PAPERS, SCHEMA, PROMPT)
score(records, GOLD, label="2회차 · LLM 출력")

In [ ]:
checked = validate(records)
score(checked, GOLD, label="2회차 · 코드 검증 후")

### 3회차 — 프롬프트 직접 쓰기: LLM 출력부터 맞게

2회차는 **코드**가 고쳤다. LLM 출력 자체는 아직 틀린다. 이번에는 스키마는 v2 그대로 두고 **프롬프트만** 고쳐서,
**코드 검증 전**에 F1 1.00을 만든다 (검증 로그에 제거·환산이 하나도 없으면 완벽).

**왜 필요한가**
- 코드는 **버리기만** 한다 → 잘못 뽑힌 값을 버릴 수는 있어도, 맞는 값을 대신 채워 넣지는 못한다
- 코드 규칙은 미리 아는 함정만 잡는다 → 새 논문에서는 프롬프트에 적은 도메인 지식이 먼저 막는다

**남은 문제** (2회차 `LLM 출력`의 틀린 것 표에서 확인)

| 문제 | 프롬프트에 넣을 것 |
|---|---|
| 포메이션 사이클 값(205, 198)이 초기 용량으로 들어옴 | **도메인 지식**: 포메이션 사이클이 무엇인지, 초기 방전 용량의 정의 |
| 논문 B 이온전도도가 S m⁻¹ 그대로 들어옴 | **도메인 지식**: 단위 환산 / **출력 형식**: 표준 단위로 쓰기 |
| 셀 이름이 엉뚱하게 들어옴 (가끔) | **출력 형식**: `material`에 무엇을 쓸지 |
| 인용값이 들어옴 (가끔) | **추출 규칙**: 인용 표시를 알아보는 법 |

**프롬프트 틀**: `[역할]` → `[도메인 지식]` → `[추출 규칙]` → `[출력 형식]`
- 한 번에 한 가지씩 넣고 실행해, 어떤 줄이 무엇을 고쳤는지 본다
- 같은 프롬프트는 저장된 결과를 다시 쓴다. 한 글자라도 바꾸면 새로 호출한다 (매번 3회)
- 다 써 본 뒤 아래 **예시 답안** 셀을 열어 비교한다

In [ ]:
SCHEMA = "v2"   # 3회차: 스키마는 그대로, 프롬프트만 직접 쓴다
PROMPT = """
[역할]
-

[도메인 지식]
-

[추출 규칙]
- 이 논문에서 직접 측정한 값만 뽑는다. 다른 논문에서 인용한 값은 뽑지 않는다.

[출력 형식]
-
"""

In [ ]:
records = extract(PAPERS, SCHEMA, PROMPT)
score(records, GOLD, label="3회차 · LLM 출력")   # 목표: 검증 전에 F1 1.00
checked = validate(records)
score(checked, GOLD, label="3회차 · 코드 검증 후")

In [ ]:
#@title 3회차 예시 답안 — 직접 써 본 뒤 열어 보기 (실행하면 답안 프롬프트로 추출·채점)
PROMPT_ANSWER = """
[역할]
너는 리튬 금속 전지 논문에서 실험값을 뽑아 데이터베이스를 만드는 연구원이다.

[도메인 지식]
- 포메이션 사이클: 본 시험 전에 낮은 C-rate(예: 0.1C)로 1–2회 돌리는 활성화 사이클이다. 이때 나온 용량은 초기 방전 용량이 아니다.
- Methods에 포메이션 C-rate가 적혀 있으면, Results에서 그 C-rate로 잰 첫 사이클 용량도 포메이션 값이다.
- 초기 방전 용량(initial_discharge_capacity): 본 시험 C-rate(예: 1C, 0.5C)에서 첫 사이클의 방전 용량이다.
- 이온전도도 단위 환산: 1 S m−1 = 10 mS cm−1, 1 S cm−1 = 1000 mS cm−1.
- "[숫자]" 인용 표시나 "previous study", "previous reports"가 붙은 값은 다른 논문의 값이다.

[추출 규칙]
- 이 논문에서 직접 측정한 값만 뽑는다. 다른 논문에서 인용한 값은 뽑지 않는다.
- 포메이션 사이클 값은 뽑지 않는다.
- 네 가지 물성(이온전도도, 초기 방전 용량, 용량 유지율, 평균 쿨롱효율) 외의 값은 뽑지 않는다.

[출력 형식]
- value와 unit은 표준 단위로 환산해 쓴다: 이온전도도 mS/cm, 용량 mAh/g, 유지율·쿨롱효율 %.
  예) 1.05 S m−1 → value 10.5, unit "mS/cm"
- material에는 그 값을 낸 셀·전해질의 이름을 본문 그대로 쓴다 (예: DEE-based cell, LiNO3-added cell, LHCE cell).
  전지 구성(Li||LFP, Li||NCM811)은 셀 이름이 아니다. 문장의 주어가 무엇인지 확인한다.
- evidence에는 그 값이 나온 원문 문장 하나를 그대로 복사한다 (원문 단위는 여기에 남는다).
"""
answer_records = extract(PAPERS, "v2", PROMPT_ANSWER)
score(answer_records, GOLD, label="3회차 · 예시 답안 LLM 출력")

### 회차별 비교

**LLM 출력**과 **코드 검증 후** 점수가 회차마다 어떻게 바뀌었나?
- 2회차: 코드가 고친다 (LLM 출력은 그대로 틀림)
- 3회차: 프롬프트가 고친다 (코드는 걸러낼 것이 없어야 한다)

In [ ]:
show_scores()

## 7. 그림 범례와 본문 셀 이름 맞추기 — 셀 단위 표

논문 A의 Fig. 2 범례는 `E1, E2, Ref.`인데, 본문의 셀 이름은 `DEE-based cell`, `baseline cell`이다.
- 캡션: "E1, E2 and Ref. denote the **DEE, DPE and DEGDME electrolytes**"
- Methods: "the corresponding cells are denoted as the **DEE-based cell**, the DPE-based cell and the **baseline cell**"
- 범례 → 전해질 → 셀 이름으로 **두 번 건너야** 연결된다

왜 필요한가: 그래프에서 읽은 값(LLMB에서는 사이클 곡선)은 **범례로만** 구분된다. 범례를 셀 이름에 이어야 본문 레코드와 한 줄(셀 단위 레코드)로 합칠 수 있다.

**해 볼 것**
1. `MATCH = "exact"`(글자가 같아야 연결)로 실행 → 표가 비는 것을 확인
2. `MATCH = "llm"`으로 바꿔 실행 → 아래 `MERGE_PROMPT`가 LLM에 들어간다. 출력 맨 위에 **실제로 보낸 프롬프트**가 찍힌다
3. 표 정확도가 1.00이 아니다. 어느 칸이 틀렸나? `MERGE_PROMPT`에 **출력 형식**을 적어 1.00을 만든다
4. 다 해 본 뒤 아래 **예시 답안** 셀과 비교한다

- 채점: 범례마다 **셀 이름 · 전해질(염 + 농도 + 용매) · 양극 · C-rate** 네 칸
- 프롬프트의 `{legends}`, `{names}`, `{paper}` 자리에는 범례 라벨, 추출 레코드의 이름, 논문 A 원문이 자동으로 들어간다
- LLM 답의 칸(`legend`, `cell_name`, `electrolyte`, `cathode`, `c_rate`)은 스키마로 정해져 있다. **칸에 무엇을 어떤 형식으로 쓸지는 프롬프트가 정한다**

In [ ]:
MATCH = "exact"   # → "llm"
MERGE_PROMPT = """
그림의 범례 라벨을 본문에 나온 셀과 연결한다.

범례 라벨: {legends}
추출한 레코드에 나온 이름 (참고): {names}

논문:
{paper}
"""
table = merge(checked, MATCH, MERGE_PROMPT)
score(table, GOLD_TABLE)

In [ ]:
#@title 7번 예시 답안 — 직접 고쳐 본 뒤 열어 보기 (실행하면 답안 프롬프트로 매칭·채점)
MERGE_PROMPT = """
그림의 범례 라벨을 본문에 나온 셀과 연결한다.
- 캡션에서 각 범례가 어떤 전해질을 뜻하는지 찾고, Methods에서 그 전해질을 쓴 셀 이름을 찾는다.
- cell_name에는 Methods에 나온 셀 이름을 그대로 쓴다. 찾지 못하면 null로 둔다.
- electrolyte에는 염과 용매를 쓴다 (예: 1.0 M LiFSI in DEE). cathode에는 양극 물질, c_rate에는 본 시험 C-rate를 쓴다.

범례 라벨: {legends}
추출한 레코드에 나온 이름 (참고): {names}

논문:
{paper}
"""
table = merge(checked, "llm", MERGE_PROMPT)
score(table, GOLD_TABLE)

## 8. 결과 내려받기

`records.csv`, `cell_table.csv`, `schema.json`, `prompt.json`(추출·매칭 프롬프트), `scores.csv`를 zip으로 묶는다.
**`records.csv`는 실습 2에서 다시 쓴다.**

In [ ]:
export(checked, table, SCHEMA, PROMPT, MERGE_PROMPT)

## 정리

- 스키마는 **무엇을**, 프롬프트는 **어떻게** 꺼낼지 정한다
- 형식이 맞아도 내용은 틀릴 수 있다 → **근거 문장**과 **이 논문 값인가**를 같이 뽑는다
- 단위 환산·범위 확인은 **코드로** 한다
- 프롬프트에 **도메인 지식**(포메이션, 단위 환산)과 **출력 형식**(표준 단위, 셀 이름)을 적으면 LLM 출력부터 맞는다 → 코드는 확인용으로 남긴다
- 범례와 셀 이름처럼 이름이 다른 것을 잇는 일은 LLM이 잘한다 → 셀 단위 레코드 완성
- 매칭도 프롬프트로 지시한다 → 어디를 보고(캡션, Methods) 무엇을 어떤 형식으로 쓸지 적는다